# Viewing and Editing Segmentations V1
Tommy Tang <br>
Oct 17th, 2026

In [ ]:
#Install dependencies
# One-time setup for the viewer, HDF5 I/O, arrays, graph cuts, and progress bars.
%pip install neuroglancer h5py numpy scipy networkx tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Libraries and paths
# Neuroglancer's Python API serves local arrays and receives browser actions.
from pathlib import Path
import json

import h5py
import networkx as nx
import neuroglancer
import numpy as np

from scipy import ndimage
from tqdm.auto import tqdm
from IPython.display import HTML, IFrame, display


# Location of your CREMI file.
DATA_PATH = Path("data/sampleA.h5")

# Generated supervoxel volume.
SUPERVOXEL_PATH = Path("state/supervoxels_xyz.npy")

# Editable graph state.
EDITS_PATH = Path("state/edits.json")

SUPERVOXEL_PATH.parent.mkdir(parents=True, exist_ok=True)

/Users/tommy/Projects/.torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Inspect hdf5 file
# Listing shapes and attributes verifies alignment and exposes voxel metadata.
with h5py.File(DATA_PATH, "r") as file:
    print("Objects inside the HDF5 file:\n")

    def describe(name, obj):
        shape = getattr(obj, "shape", None)
        dtype = getattr(obj, "dtype", None)

        print(name)
        print("  shape:", shape)
        print("  dtype:", dtype)
        print("  attributes:", dict(obj.attrs))

    file.visititems(describe)

Objects inside the HDF5 file:

__DATA_TYPES__
  shape: None
  dtype: None
  attributes: {}
__DATA_TYPES__/Enum_Boolean
  shape: None
  dtype: bool
  attributes: {}
__DATA_TYPES__/String_VariableLength
  shape: None
  dtype: object
  attributes: {}
annotations
  shape: None
  dtype: None
  attributes: {}
annotations/comments
  shape: None
  dtype: None
  attributes: {}
annotations/comments/comments
  shape: (17,)
  dtype: object
  attributes: {}
annotations/comments/target_ids
  shape: (17,)
  dtype: uint64
  attributes: {}
annotations/ids
  shape: (432,)
  dtype: uint64
  attributes: {}
annotations/locations
  shape: (432, 3)
  dtype: float32
  attributes: {}
annotations/presynaptic_site
  shape: None
  dtype: None
  attributes: {}
annotations/presynaptic_site/partners
  shape: (216, 2)
  dtype: uint64
  attributes: {}
annotations/types
  shape: (432,)
  dtype: object
  attributes: {}
volumes
  shape: None
  dtype: None
  attributes: {}
volumes/labels
  shape: None
  dtype: None
  attr

In [ ]:
#Load a small crop
# CREMI is stored as ZYX; the viewer-facing arrays are converted to XYZ below.
with h5py.File(DATA_PATH, "r") as file:
    raw_dataset = file["volumes/raw"]
    label_dataset = file["volumes/labels/neuron_ids"]

    print("Full raw shape:", raw_dataset.shape)
    print("Full label shape:", label_dataset.shape)
    print("Raw attributes:", dict(raw_dataset.attrs))

    # Get the actual resolution stored in the dataset.
    # CREMI normally reports this in Z, Y, X order in nanometers.
    if "resolution" not in raw_dataset.attrs:
        raise KeyError(
            "The raw dataset has no 'resolution' attribute. "
            "Inspect the printed attributes before continuing."
        )

    resolution_zyx = np.asarray(
        raw_dataset.attrs["resolution"],
        dtype=float,
    )

    # Choose a crop near the center of the dataset.
    # Takes the first 32 Z slices and centers 384x384 in Y/X.
    # With 4x4x40 nm XYZ voxels, this is about 1.536x1.536x1.280 µm.
    z_size = min(32, raw_dataset.shape[0])
    y_size = min(384, raw_dataset.shape[1])
    x_size = min(384, raw_dataset.shape[2])

    z0 = 0
    y0 = max(0, raw_dataset.shape[1] // 2 - y_size // 2)
    x0 = max(0, raw_dataset.shape[2] // 2 - x_size // 2)

    crop_zyx = (
        slice(z0, z0 + z_size),
        slice(y0, y0 + y_size),
        slice(x0, x0 + x_size),
    )

    # Reading with slices converts these HDF5 datasets into NumPy arrays.
    raw_zyx = np.asarray(raw_dataset[crop_zyx])
    labels_zyx = np.asarray(label_dataset[crop_zyx])


# Convert array order from ZYX to XYZ.
# This changes axis order only; it does not resample or interpolate voxels.
raw_xyz = np.ascontiguousarray(raw_zyx.transpose(2, 1, 0))
labels_xyz = np.ascontiguousarray(labels_zyx.transpose(2, 1, 0))

# Reverse the resolution for the same reason.
resolution_xyz = resolution_zyx[::-1]

# Preserve the crop's position in the full dataset.
voxel_offset_xyz = np.asarray([x0, y0, z0], dtype=int)

print("Crop shape in XYZ order:", raw_xyz.shape)
print("Resolution in XYZ order:", resolution_xyz, "nm")
print("Voxel offset in XYZ order:", voxel_offset_xyz)
print("Approximate memory:")
print(f"  raw:    {raw_xyz.nbytes / 1e6:.1f} MB")
print(f"  labels: {labels_xyz.nbytes / 1e6:.1f} MB")

Full raw shape: (125, 1250, 1250)
Full label shape: (125, 1250, 1250)
Raw attributes: {'resolution': array([40.,  4.,  4.])}
Crop shape in XYZ order: (384, 384, 32)
Resolution in XYZ order: [ 4.  4. 40.] nm
Voxel offset in XYZ order: [433 433   0]
Approximate memory:
  raw:    4.7 MB
  labels: 37.7 MB


In [ ]:
#Start neuroglancer
# Python owns the data/state; the browser renders it and sends interactions back.
# CoordinateSpace tells Neuroglancer what every array axis represents.
# Dataset scales preserve CREMI's anisotropic physical dimensions.
dimensions = neuroglancer.CoordinateSpace(
    names=["x", "y", "z"],
    units=["nm", "nm", "nm"],
    scales=resolution_xyz,
)

# A Viewer starts a small local web server in the background.
viewer = neuroglancer.Viewer()

viewer_url = viewer.get_viewer_url()

display(
    HTML(
        f'<a href="{viewer_url}" target="_blank">'
        "Open the Neuroglancer client in a new tab"
        "</a>"
    )
)

print(viewer_url)

http://127.0.0.1:57580/v/82518f46316cd7c91c89a657fd6d829334b01f9e/


In [ ]:
#Display EM volume
# LocalVolume serves requested chunks instead of embedding the crop in the page.
raw_volume = neuroglancer.LocalVolume(
    data=raw_xyz,
    dimensions=dimensions,
    voxel_offset=voxel_offset_xyz,
    volume_type="image",
)

with viewer.txn() as state:
    state.layers["em"] = neuroglancer.ImageLayer(
        source=raw_volume,
        opacity=1.0,
        shader="""
void main() {
    emitGrayscale(toNormalized(getDataValue()));
}
""",
    )

    # Start the viewer near the middle of our crop.
    state.position = (
        voxel_offset_xyz
        + np.asarray(raw_xyz.shape, dtype=float) / 2
    )

    state.layout = "xy"

In [ ]:
#Add segmentations
# This immutable reference segmentation defines the initial neuron grouping.
original_label_volume = neuroglancer.LocalVolume(
    data=labels_xyz,
    dimensions=dimensions,
    voxel_offset=voxel_offset_xyz,
    volume_type="segmentation",
)

with viewer.txn() as state:
    state.layers["original_labels"] = neuroglancer.SegmentationLayer(
        source=original_label_volume,

        # Selected objects appear more opaque.
        selected_alpha=0.8,

        # Other objects remain partially transparent.
        not_selected_alpha=0.25,

        object_alpha=0.5,
    )

In [63]:
#Create empty live supervoxel layer
# This dense ID volume stays fixed after preprocessing; the graph is edited later.
# uint32 is sufficient for the number of supervoxels in this crop.
supervoxels_xyz = np.zeros(labels_xyz.shape, dtype=np.uint32)

# Integer LocalVolume uses _neuroglancer.OnDemandObjectMeshGenerator:
# marching-cubes surface extraction followed by Neuroglancer/WebGL rendering.
supervoxel_volume = neuroglancer.LocalVolume(
    data=supervoxels_xyz,
    dimensions=dimensions,
    voxel_offset=voxel_offset_xyz,
    volume_type="segmentation",
)

with viewer.txn() as state:
    state.layers["supervoxels"] = neuroglancer.SegmentationLayer(
        source=supervoxel_volume,
        selected_alpha=0.8,
        not_selected_alpha=0.35,
        object_alpha=0.5,
    )

    # Temporarily hide the original segmentation so we can watch the
    # supervoxel construction.
    state.layers["original_labels"].visible = False

In [64]:
def iter_blocks(shape, block_shape):
    """Generate 3D NumPy slices covering an array."""

    for x0 in range(0, shape[0], block_shape[0]):
        for y0 in range(0, shape[1], block_shape[1]):
            for z0 in range(0, shape[2], block_shape[2]):
                yield (
                    slice(x0, min(x0 + block_shape[0], shape[0])),
                    slice(y0, min(y0 + block_shape[1], shape[1])),
                    slice(z0, min(z0 + block_shape[2], shape[2])),
                )


# 64x64x8 voxels is about 256x256x320 nm at CREMI's 4x4x40 nm scale.
# The near-isotropic physical block balances split detail against graph size.
def construct_supervoxels(
    labels,
    output,
    block_shape=(64, 64, 8),
    update_every=4,
):
    """
    Subdivide each original neuron into small connected regions.

    Parameters
    ----------
    labels:
        Original XYZ neuron-label array.

    output:
        XYZ uint32 array that will receive supervoxel IDs.

    block_shape:
        Size of each XYZ block. Z is smaller because CREMI voxels
        are much thicker along Z.

    update_every:
        Ask Neuroglancer to refresh after this many blocks.
    """

    # Six-connectivity means voxels connect through faces, not merely
    # through corners.
    connectivity = ndimage.generate_binary_structure(
        rank=3,
        connectivity=1,
    )

    blocks = list(iter_blocks(labels.shape, block_shape))
    next_supervoxel_id = 1

    # Treat zero as background.
    ignored_ids = {0}

    # Many segmentation datasets use the maximum integer as an ignore label.
    if np.issubdtype(labels.dtype, np.integer):
        ignored_ids.add(int(np.iinfo(labels.dtype).max))

    for block_number, region in enumerate(tqdm(blocks)):
        label_block = labels[region]

        # This is a view into the output array, so modifying it modifies
        # supervoxels_xyz directly.
        output_block = output[region]

        for neuron_id in np.unique(label_block):
            if int(neuron_id) in ignored_ids:
                continue

            # Find connected pieces of this neuron inside the current block.
            connected_pieces, number_of_pieces = ndimage.label(
                label_block == neuron_id,
                structure=connectivity,
            )

            for local_piece_id in range(1, number_of_pieces + 1):
                output_block[
                    connected_pieces == local_piece_id
                ] = next_supervoxel_id

                next_supervoxel_id += 1

        # LocalVolume holds the same mutable NumPy array. Calling invalidate()
        # tells open clients to request the changed chunks again.
        if block_number % update_every == 0:
            supervoxel_volume.invalidate()

    # Final refresh.
    supervoxel_volume.invalidate()

    return next_supervoxel_id - 1


number_of_supervoxels = construct_supervoxels(
    labels=labels_xyz,
    output=supervoxels_xyz,
)

print("Created", number_of_supervoxels, "supervoxels")

# Saved graph edges depend on these deterministic supervoxel IDs.
np.save(SUPERVOXEL_PATH, supervoxels_xyz)

100%|██████████| 144/144 [00:01<00:00, 89.14it/s]

Created 2006 supervoxels


In [65]:
def build_supervoxel_graph(
    supervoxels,
    original_labels,
    voxel_size,
):
    """Build an undirected adjacency graph."""
    # Nodes are supervoxels; edges are same-neuron face contacts.

    graph = nx.Graph()

    # Add every nonzero supervoxel as a graph node.
    node_ids = np.unique(supervoxels)
    node_ids = node_ids[node_ids != 0]

    graph.add_nodes_from(int(node_id) for node_id in node_ids)

    # Examine neighbors along X, Y, and Z.
    for axis in tqdm(range(3), desc="Scanning adjacency"):
        left_slice = [slice(None)] * 3
        right_slice = [slice(None)] * 3

        left_slice[axis] = slice(0, -1)
        right_slice[axis] = slice(1, None)

        left_slice = tuple(left_slice)
        right_slice = tuple(right_slice)

        left_sv = supervoxels[left_slice]
        right_sv = supervoxels[right_slice]

        left_labels = original_labels[left_slice]
        right_labels = original_labels[right_slice]

        valid = (
            # Both voxels belong to supervoxels.
            (left_sv != 0)
            & (right_sv != 0)

            # The two supervoxels are different.
            & (left_sv != right_sv)

            # Initially connect only pieces of the same supplied neuron.
            & (left_labels == right_labels)
        )

        if not np.any(valid):
            continue

        first = left_sv[valid].astype(np.uint64)
        second = right_sv[valid].astype(np.uint64)

        # Store every pair in a consistent order.
        low = np.minimum(first, second)
        high = np.maximum(first, second)

        pairs = np.stack([low, high], axis=1)

        # Count how many voxel faces each pair shares.
        unique_pairs, face_counts = np.unique(
            pairs,
            axis=0,
            return_counts=True,
        )

        # Physical contact area is the min-cut capacity, so narrow
        # interfaces are cheaper to cut than broad interfaces.
        # A face perpendicular to `axis` has dimensions equal to the
        # other two voxel dimensions.
        other_axes = [i for i in range(3) if i != axis]
        physical_face_area = float(
            voxel_size[other_axes[0]]
            * voxel_size[other_axes[1]]
        )

        for pair, face_count in zip(unique_pairs, face_counts):
            u, v = int(pair[0]), int(pair[1])
            added_capacity = float(face_count * physical_face_area)

            if graph.has_edge(u, v):
                graph[u][v]["capacity"] += added_capacity
            else:
                graph.add_edge(
                    u,
                    v,
                    capacity=added_capacity,
                    kind="initial",
                )

    return graph


initial_graph = build_supervoxel_graph(
    supervoxels=supervoxels_xyz,
    original_labels=labels_xyz,
    voxel_size=resolution_xyz,
)

print("Graph nodes:", initial_graph.number_of_nodes())
print("Graph edges:", initial_graph.number_of_edges())
print(
    "Initial connected components:",
    nx.number_connected_components(initial_graph),
)

Scanning adjacency: 100%|██████████| 3/3 [00:00<00:00, 13.37it/s]

Graph nodes: 2006
Graph edges: 1256
Initial connected components: 1171


In [66]:
#Display graph components as objects
# Each connected graph component becomes one logical Neuroglancer object.
def components_and_roots(graph):
    """
    Return Neuroglancer equivalence groups and a node-to-root mapping.

    Neuroglancer uses the smallest ID in an equivalence group as its
    representative, so we do the same.
    """

    components = [
        sorted(int(node) for node in component)
        for component in nx.connected_components(graph)
    ]

    root_for_node = {
        node: min(component)
        for component in components
        for node in component
    }

    return components, root_for_node


components, root_for_node = components_and_roots(initial_graph)

with viewer.txn() as state:
    state.layers["objects"] = neuroglancer.SegmentationLayer(
        source=supervoxel_volume,
        # Equivalences merge display identity without rewriting voxel IDs.
        equivalences=components,
        selected_alpha=0.9,
        not_selected_alpha=0.25,
        object_alpha=0.7,
    )

    # Hide the diagnostic layers. They remain available in the layer panel.
    state.layers["original_labels"].visible = False
    state.layers["supervoxels"].visible = False

    # Show an XY slice beside a 3D mesh view.
    state.layout = neuroglancer.row_layout([
        neuroglancer.LayerGroupViewer(
            layout="xy",
            layers=["em", "objects", "original_labels", "supervoxels"],
        ),
        neuroglancer.LayerGroupViewer(
            layout="3d",
            layers=["objects"],
        ),
    ])

In [68]:
#Interactive editor
# The graph is editable state; the dense supervoxel array remains unchanged.
class NotebookSegmentationEditor:
    def __init__(
        self,
        viewer,
        initial_graph,
        dimensions,
        state_path,
        metadata,
    ):
        self.viewer = viewer
        self.state_path = Path(state_path)
        self.metadata = metadata

        # Start with a copy so the original graph remains unchanged.
        self.graph = initial_graph.copy()

        self.history = []

        # The two user-selected supervoxels.
        self.seed_a = None
        self.seed_b = None

        # Mouse positions used to display seed annotations.
        self.seed_a_position = None
        self.seed_b_position = None

        # Restore saved edits if a compatible save file exists.
        # This recreates graph connectivity before equivalences are refreshed.
        self.load_if_available()

        # Add two annotation layers so selected seeds are visible.
        # Point annotations belong to LocalAnnotationLayer, not SegmentationLayer.
        with self.viewer.txn() as state:
            state.layers["seed_a"] = neuroglancer.LocalAnnotationLayer(
                dimensions=dimensions,
                annotation_color="#00ff00",
            )

            state.layers["seed_b"] = neuroglancer.LocalAnnotationLayer(
                dimensions=dimensions,
                annotation_color="#ff00ff",
            )

        # Connect Neuroglancer action names to Python methods.
        self.viewer.actions.add("choose-seed-a", self.choose_seed_a)
        self.viewer.actions.add("choose-seed-b", self.choose_seed_b)
        self.viewer.actions.add("merge-seeds", self.merge)
        self.viewer.actions.add("split-seeds", self.split)
        self.viewer.actions.add("clear-seeds", self.clear_seeds)

        # Connect browser input events to those action names.
        with self.viewer.config_state.txn() as config:
            config.input_event_bindings.data_view[
                "shift+mousedown0"
            ] = "choose-seed-a"

            config.input_event_bindings.data_view[
                "control+mousedown0"
            ] = "choose-seed-b"

            config.input_event_bindings.data_view["keym"] = "merge-seeds"
            config.input_event_bindings.data_view["keys"] = "split-seeds"
            config.input_event_bindings.data_view["keyc"] = "clear-seeds"

            config.status_messages["help"] = (
                "Shift-click: seed A | "
                "Control-click: seed B | "
                "M: merge | S: split | C: clear"
            )

        self.refresh_viewer()
        self.set_status("Editor ready")

    def selected_supervoxel(self, action_state):
        """Get the raw supervoxel ID underneath the mouse."""

        selected = action_state.selected_values.get("objects")

        if selected is None:
            return None

        value = selected.value

        # When equivalences are active, Neuroglancer can return both:
        #   key   = raw supervoxel ID
        #   value = logical component ID
        if isinstance(value, neuroglancer.SegmentIdMapEntry):
            return int(value.key)

        if value is None:
            return None

        return int(value)

    def choose_seed_a(self, action_state):
        supervoxel_id = self.selected_supervoxel(action_state)

        if supervoxel_id is None:
            self.set_status("Seed A was not placed on a segmentation")
            return

        self.seed_a = supervoxel_id
        self.seed_a_position = np.asarray(
            action_state.mouse_voxel_coordinates,
            dtype=float,
        )

        self.update_seed_annotations()
        self.refresh_viewer()
        self.set_status(f"Seed A = supervoxel {self.seed_a}")

    def choose_seed_b(self, action_state):
        supervoxel_id = self.selected_supervoxel(action_state)

        if supervoxel_id is None:
            self.set_status("Seed B was not placed on a segmentation")
            return

        self.seed_b = supervoxel_id
        self.seed_b_position = np.asarray(
            action_state.mouse_voxel_coordinates,
            dtype=float,
        )

        self.update_seed_annotations()
        self.refresh_viewer()
        self.set_status(f"Seed B = supervoxel {self.seed_b}")

    def update_seed_annotations(self):
        """Show seed A and seed B as colored points."""

        with self.viewer.txn() as state:
            if self.seed_a_position is None:
                state.layers["seed_a"].annotations = []
            else:
                state.layers["seed_a"].annotations = [
                    neuroglancer.PointAnnotation(
                        id="seed-a",
                        point=self.seed_a_position,
                        description=f"A: supervoxel {self.seed_a}",
                    )
                ]

            if self.seed_b_position is None:
                state.layers["seed_b"].annotations = []
            else:
                state.layers["seed_b"].annotations = [
                    neuroglancer.PointAnnotation(
                        id="seed-b",
                        point=self.seed_b_position,
                        description=f"B: supervoxel {self.seed_b}",
                    )
                ]

    def selected_nodes(self):
        """Return whichever seeds currently exist."""

        return {
            node
            for node in [self.seed_a, self.seed_b]
            if node is not None
        }

    def refresh_viewer(self):
        """Recompute graph components and update Neuroglancer."""

        # Recompute graph components; no dense voxel relabeling is needed.
        components, self.root_for_node = components_and_roots(
            self.graph
        )

        # Selecting a component root tells Neuroglancer to request and
        # display its mesh in the 3D panel.
        visible_roots = {
            self.root_for_node[node]
            for node in self.selected_nodes()
            if node in self.root_for_node
        }

        with self.viewer.txn() as state:
            objects = state.layers["objects"]

            # This updates the logical segmentation overlay.
            objects.equivalences = components

            # These components are selected, so their meshes appear.
            # Selection triggers on-demand meshing and a 3-D redraw.
            objects.segments = visible_roots

    def merge(self, action_state=None):
        """Merge the objects containing seed A and seed B."""

        del action_state

        if self.seed_a is None or self.seed_b is None:
            self.set_status("Choose both seed A and seed B first")
            return

        root_a = self.root_for_node[self.seed_a]
        root_b = self.root_for_node[self.seed_b]

        if root_a == root_b:
            self.set_status("A and B already belong to the same object")
            return

        # Adding one edge joins the two connected components.
        # No voxel values need to change for a merge.
        self.graph.add_edge(
            self.seed_a,
            self.seed_b,

            # A small capacity makes this manual merge easy to undo
            # with a future minimum cut.
            capacity=1.0,
            kind="manual-merge",
        )

        self.history.append({
            "operation": "merge",
            "a": self.seed_a,
            "b": self.seed_b,
        })

        self.save()
        self.refresh_viewer()

        self.set_status(
            f"Merged objects through edge "
            f"{self.seed_a}–{self.seed_b}"
        )

    def split(self, action_state=None):
        """
        Split one object by separating seed A from seed B.

        The minimum-cut algorithm finds a low-capacity set of graph
        edges whose removal separates the two seeds.
        """

        del action_state

        if self.seed_a is None or self.seed_b is None:
            self.set_status("Choose both split seeds first")
            return

        root_a = self.root_for_node[self.seed_a]
        root_b = self.root_for_node[self.seed_b]

        if root_a != root_b:
            self.set_status(
                "A and B already belong to different objects"
            )
            return

        # Work only on the object containing seed A.
        component_nodes = nx.node_connected_component(
            self.graph,
            self.seed_a,
        )

        component_graph = self.graph.subgraph(
            component_nodes
        ).copy()

        # Edge capacity is shared physical face area from graph construction.
        cut_value, partitions = nx.minimum_cut(
            component_graph,
            self.seed_a,
            self.seed_b,
            capacity="capacity",
        )

        left, right = partitions

        # Find all original graph edges crossing the cut.
        cut_edges = []

        for u in left:
            for v in self.graph.neighbors(u):
                if v in right:
                    cut_edges.append(
                        tuple(sorted((int(u), int(v))))
                    )

        cut_edges = sorted(set(cut_edges))

        self.graph.remove_edges_from(cut_edges)

        self.history.append({
            "operation": "split",
            "a": self.seed_a,
            "b": self.seed_b,
            "cut_value": float(cut_value),
            "removed_edges": [
                list(edge) for edge in cut_edges
            ],
        })

        self.save()
        self.refresh_viewer()

        self.set_status(
            f"Split object by removing "
            f"{len(cut_edges)} graph edges"
        )

    def clear_seeds(self, action_state=None):
        del action_state

        self.seed_a = None
        self.seed_b = None
        self.seed_a_position = None
        self.seed_b_position = None

        self.update_seed_annotations()
        self.refresh_viewer()
        self.set_status("Selections cleared")

    def set_status(self, message):
        """Display a message along the bottom of Neuroglancer."""

        print(message)

        with self.viewer.config_state.txn() as config:
            config.status_messages["editor"] = message

    def save(self):
        """Save the entire current graph atomically."""

        edges = []

        for u, v, attributes in self.graph.edges(data=True):
            edges.append({
                "u": int(u),
                "v": int(v),
                "capacity": float(
                    attributes.get("capacity", 1.0)
                ),
                "kind": attributes.get("kind", "unknown"),
            })

        # The edge list captures added merge edges and surviving split edges.
        data = {
            "version": 1,
            "metadata": self.metadata,
            "edges": edges,
            "history": self.history,
        }

        self.state_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        temporary_path = self.state_path.with_suffix(".tmp")

        temporary_path.write_text(
            json.dumps(data, indent=2)
        )

        # Replacing the file is safer than writing directly to it.
        temporary_path.replace(self.state_path)

    def load_if_available(self):
        """Restore a previously saved graph if it is compatible."""

        if not self.state_path.exists():
            print("No saved edits found; using the initial graph")
            return

        data = json.loads(self.state_path.read_text())

        if data.get("metadata") != self.metadata:
            raise ValueError(
                "The saved edits were created for a different "
                "crop or supervoxel configuration."
            )

        # Recreate all deterministic nodes, then restore saved connectivity.
        restored_graph = nx.Graph()
        restored_graph.add_nodes_from(self.graph.nodes)

        for edge in data["edges"]:
            restored_graph.add_edge(
                int(edge["u"]),
                int(edge["v"]),
                capacity=float(edge["capacity"]),
                kind=edge.get("kind", "unknown"),
            )

        self.graph = restored_graph
        self.history = data.get("history", [])

        print(
            f"Restored {len(self.history)} saved edits "
            f"from {self.state_path}"
        )

In [71]:
#Start editor
# Metadata prevents saved supervoxel IDs from being applied to another crop.
metadata = {
    "source": str(DATA_PATH),
    "crop_zyx": [
        [z0, z0 + z_size],
        [y0, y0 + y_size],
        [x0, x0 + x_size],
    ],
    "block_shape_xyz": [64, 64, 8],
    "supervoxel_shape_xyz": list(supervoxels_xyz.shape),
    "number_of_supervoxels": int(number_of_supervoxels),
}

editor = NotebookSegmentationEditor(
    viewer=viewer,
    initial_graph=initial_graph,
    dimensions=dimensions,
    state_path=EDITS_PATH,
    metadata=metadata,
)

Restored 2 saved edits from state/edits.json
Editor ready


Seed B = supervoxel 1008
Seed B = supervoxel 1008
Seed A = supervoxel 710
Seed A = supervoxel 710
Seed A = supervoxel 1141
Seed A = supervoxel 1141
Seed A = supervoxel 421
Seed A = supervoxel 421
Seed A = supervoxel 1097
Seed A = supervoxel 1097
Seed A = supervoxel 710
Seed A = supervoxel 710
